In [1]:
import os
import sys
import signal
import yaml

import torch
from transformers import AutoTokenizer

from lightning.pretokenized_pure_text_dataset import DataModule
from utils.data_utils import Struct

/home/huang717/.conda/envs/test/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
config_path = "/home/huang717/DRAGN/IRM/injectable-alignment-model/configs/Llama-2-7b-chat-hf_tiny_shakespeare_31_training.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)
config = Struct(**config)
tokenizer = AutoTokenizer.from_pretrained(config.model_name)
tokenizer.pad_token = tokenizer.eos_token
config.pad_id = tokenizer.pad_token_id

In [3]:
tokenizer.pad_token

'</s>'

In [4]:
tokenizer.decode(13)

'\n'

In [5]:
dm = DataModule(config, tokenizer)
dm.setup()

Loaded tokenized data from /home/huang717/DRAGN/IRM/injectable-alignment-model/datasets/TinyShakespeare//split/train.pt
pad_tok: 2
Loaded tokenized data from /home/huang717/DRAGN/IRM/injectable-alignment-model/datasets/TinyShakespeare//split/val.pt
pad_tok: 2


In [6]:
torch.cuda.is_available()

True

In [7]:
loader = dm.train_dataloader()

In [8]:
# Get a single batch
batch = next(iter(loader))

# Inspect the batch
print("Number of items in batch:", len(batch))
print("\nShapes:")
for i, item in enumerate(batch):
    print(f"Item {i} shape:", item.shape)
    print(f"Item {i} dtype:", item.dtype)
    print(f"Sample values:\n", item[0][:10])  # First sequence, first 10 tokens
    print()

# If you want to see multiple batches:
for i, batch in enumerate(loader):
    if i >= 3:  # Look at first 3 batches
        break
    print(f"\nBatch {i}:")
    print("Shapes:", [item.shape for item in batch])

Number of items in batch: 3

Shapes:
Item 0 shape: torch.Size([8, 4096])
Item 0 dtype: torch.int64
Sample values:
 tensor([23580, 29889,    13,    13, 29933,  1430, 24898,  5265, 29949, 29901])

Item 1 shape: torch.Size([8, 4096])
Item 1 dtype: torch.float32
Sample values:
 tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

Item 2 shape: torch.Size([8, 4096])
Item 2 dtype: torch.int64
Sample values:
 tensor([23580, 29889,    13,    13, 29933,  1430, 24898,  5265, 29949, 29901])


Batch 0:
Shapes: [torch.Size([8, 4096]), torch.Size([8, 4096]), torch.Size([8, 4096])]

Batch 1:
Shapes: [torch.Size([8, 4096]), torch.Size([8, 4096]), torch.Size([8, 4096])]

Batch 2:
Shapes: [torch.Size([8, 4096]), torch.Size([8, 4096]), torch.Size([8, 4096])]


In [10]:
batch

(tensor([[   13,  5965,   557,  ...,  2816,  1683,   263],
         [29915, 29879,   727,  ...,  5764,   297,   263],
         [29936,   322,    13,  ..., 29881,  3271, 29892],
         ...,
         [  787,   310,   278,  ..., 23623, 17952,  2301],
         [ 9841,   694,  2253,  ...,   275,  4322,   333],
         [  297,  1009,  1422,  ...,   590,  9184, 29877]]),
 tensor([[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]]),
 tensor([[   13,  5965,   557,  ...,  2816,  1683,   263],
         [29915, 29879,   727,  ...,  5764,   297,   263],
         [29936,   322,    13,  ..., 29881,  3271, 29892],
         ...,
         [  787,   310,   278,  ..., 23623, 17952,  2301],
         [ 9841,   694,  2253,  ...,   275,  4322,   333],
         [  297,  1009,  1422,  ...,   590,  9184, 29877]])

In [13]:
def analyze_token_frequencies(tokenizer, dataset):
    """Analyze token frequencies in the dataset with more detailed output"""
    token_counts = {}
    total_tokens = 0
    sequence_starts = []  # Track what tokens commonly start sequences
    sequence_ends = []    # Track what tokens commonly end sequences
    
    # Sample some sequences from the dataset
    num_samples = min(1000, len(dataset))
    for i in range(num_samples):
        x, _ = dataset[i]
        
        # Track start and end tokens
        if len(x) > 0:
            sequence_starts.append(x[0])
            sequence_ends.append(x[-1])
            
        # Count all tokens
        for token in x:
            token_counts[token] = token_counts.get(token, 0) + 1
            total_tokens += 1
    
    # Print detailed analysis
    print("\n=== Token Frequency Analysis ===")
    print(f"Total tokens analyzed: {total_tokens}")
    print(f"Unique tokens found: {len(token_counts)}")
    
    print("\n=== Most Common Tokens ===")
    most_common = sorted(token_counts.items(), key=lambda x: x[1], reverse=True)[:10]
    for token_id, count in most_common:
        percentage = (count / total_tokens) * 100
        decoded = tokenizer.decode([token_id])
        print(f"Token ID: {token_id}, Count: {count}, Percentage: {percentage:.2f}%, Decoded: {repr(decoded)}")
    
    print("\n=== Sequence Start Tokens ===")
    start_counts = {}
    for token in sequence_starts:
        start_counts[token] = start_counts.get(token, 0) + 1
    most_common_starts = sorted(start_counts.items(), key=lambda x: x[1], reverse=True)[:5]
    for token_id, count in most_common_starts:
        percentage = (count / len(sequence_starts)) * 100
        decoded = tokenizer.decode([token_id])
        print(f"Token ID: {token_id}, Count: {count}, Percentage: {percentage:.2f}%, Decoded: {repr(decoded)}")

    print("\n=== Sequence End Tokens ===")
    end_counts = {}
    for token in sequence_ends:
        end_counts[token] = end_counts.get(token, 0) + 1
    most_common_ends = sorted(end_counts.items(), key=lambda x: x[1], reverse=True)[:5]
    for token_id, count in most_common_ends:
        percentage = (count / len(sequence_ends)) * 100
        decoded = tokenizer.decode([token_id])
        print(f"Token ID: {token_id}, Count: {count}, Percentage: {percentage:.2f}%, Decoded: {repr(decoded)}")

In [14]:
analyze_token_frequencies(tokenizer, dm.train_dataset)


=== Token Frequency Analysis ===
Total tokens analyzed: 4096000
Unique tokens found: 1266

=== Most Common Tokens ===
Token ID: 13, Count: 461207, Percentage: 11.26%, Decoded: '\n'
Token ID: 29892, Count: 213857, Percentage: 5.22%, Decoded: ','
Token ID: 29901, Count: 113942, Percentage: 2.78%, Decoded: ':'
Token ID: 278, Count: 98214, Percentage: 2.40%, Decoded: 'the'
Token ID: 29889, Count: 81662, Percentage: 1.99%, Decoded: '.'
Token ID: 29915, Count: 72169, Percentage: 1.76%, Decoded: "'"
Token ID: 366, Count: 53578, Percentage: 1.31%, Decoded: 'you'
Token ID: 3308, Count: 50820, Percentage: 1.24%, Decoded: 'US'
Token ID: 29902, Count: 41041, Percentage: 1.00%, Decoded: 'I'
Token ID: 29879, Count: 40728, Percentage: 0.99%, Decoded: 's'

=== Sequence Start Tokens ===
Token ID: 13, Count: 118, Percentage: 11.80%, Decoded: '\n'
Token ID: 29892, Count: 50, Percentage: 5.00%, Decoded: ','
Token ID: 29901, Count: 36, Percentage: 3.60%, Decoded: ':'
Token ID: 29889, Count: 26, Percentage